# Question
* Farhad is a strange person who has n pet tigers and n² cages. Each bear i has a positive age aᵢ and a size sᵢ (no two tigers have the same age or size). 
Each cage j has a positive capacity cⱼ 
* and a specific distance dⱼ from Farhad’s bedroom (no two cages have the same capacity or the same distance). Farhad must place each bear in a separate cage.

Conditions for placing the tigers in cages:

* Farhad likes older tigers more and wants them to be placed closer to his bedroom. That means:
* If bear x is younger than bear y, then bear x must be placed in a cage that is farther from the cage of bear y.
* If a bear is placed in cage j with capacity cⱼ, and sᵢ > cⱼ, then the discomfort of the bear will be equal to sᵢ - cⱼ. Otherwise, the bear will not be uncomfortable.

Provide an O(n³) algorithm that assigns tigers to cages in a way that the above constraint is satisfied and the total discomfort of the tigers is minimized.        

# Answer
### **We have:**

* n tigers: [(a₁, s₁), ..., (aₙ, sₙ)] (age, size) sorted by ages.
* n² cages: [(c₁, d₁), ..., (cₙ², dₙ²)] (capacity, distance) sorted by distance.
* discomfort = max(0, sᵢ - cⱼ) if bear i is placed in cage j.


### **Idea:**
    
To model this, we use a Dynamic Programming (DP) approach:

1. `dp[i][j] = minimum discomfort when we assign the first i tigers to the first j cages.`
2. **Initialization:** dp[0][0] = 0 (no tigers assigned → no discomfort)
3. For each tiger i = 1 to n, we consider placing it in cages j = i to n² - (n - i), so there are always enough cages left for the remaining tigers. 
   
   For each valid pair (i, j):
        
    * `Let k = j - 1 (possible cage index for the (i-1)th tiger)`
    * `Let discomfort = max(0, sᵢ - cⱼ)`
    * `dp[i][j] = min(dp[i][j], dp[i-1][k] + discomfort) for all k from i-1 to j-1`

    To make this efficient, we maintain the minimum of dp[i-1][k] as min_prev and update it while traversing j. (`dp[i][j] = min_prev + max(0, sᵢ - cⱼ)`)

4. The final answer is min(dp[n][j]) for all j in [n, n²], which represents placing all tigers with valid spacing.
5. **Backtrace:** We maintain a choice table where choice[i][j] stores the best previous cage index k used to reach state dp[i][j].


### **Complexity Analysis**

* Number of tigers: n
* Number of cages: n²
* For each tiger i (1 to n), and each cage j (i to n² - (n - i)):
    
    → We look at only the previous best dp[i-1][k] via prefix min tracking.
    
    → Each dp[i][j] is computed in O(1) using min_prev.

* Total DP states: O(n × n²) = O(n³)
* Thus, the total complexity becomes O(n³).

In [42]:
from dataclasses import dataclass
import random

@dataclass
class Tiger:
    age: int
    size: int

@dataclass
class Cage:
    capacity: int
    distance: int

def min_discomfort(tigers, cages):
    n = len(tigers)
    tigers.sort(key=lambda x: -x.age)
    cages.sort(key=lambda x: x.distance)
    
    # Initialize DP table
    dp = [[float('inf') for _ in range(n*n + 1)] for _ in range(n + 1)]
    choice = [[-1 for _ in range(n*n + 1)] for _ in range(n + 1)]
    dp[0][0] = 0
    
    for i in range(1, n + 1):
        min_prev = float('inf')
        min_prev_index = -1

        # The i-th tiger can be placed in cages from i to n² - (n - i)
        for j in range(i, n*n - (n - i) + 1):
        
            # Update min_prev to be the min of dp[i-1][i-1 ... j-1]
            if j - 1 >= i - 1:
                if dp[i-1][j-1] < min_prev:
                    min_prev = dp[i-1][j-1]
                    min_prev_index = j - 1

                # min_prev = min(min_prev, dp[i-1][j-1])
        
            # Compute discomfort for tiger i in cage j
            discomfort = max(tigers[i-1].size - cages[j-1].capacity, 0)
            dp[i][j] = min_prev + discomfort
            choice[i][j] = min_prev_index

    # Find the best final cage index for last tiger
    min_total = float('inf')
    last_cage = -1
    for j in range(n, n*n + 1):
        if dp[n][j] < min_total:
            min_total = dp[n][j]
            last_cage = j

    # Backtrack to find assignments
    assignment = [None for _ in range(n)]
    i = n
    j = last_cage
    while i > 0:
        assignment[i-1] = j - 1  # store actual cage index (0-based)
        j = choice[i][j]
        i -= 1

    print(dp)

    return min_total, assignment, tigers, cages


In [43]:
if __name__ == "__main__":

    tigers = [
        Tiger(age=10, size=10),
        Tiger(age=8, size=3),
        Tiger(age=6, size=6),
    ]

    cages = [
        Cage(capacity=2, distance=30),
        Cage(capacity=4, distance=10),
        Cage(capacity=6, distance=20),
        Cage(capacity=5, distance=40),
        Cage(capacity=7, distance=50),
        Cage(capacity=3, distance=60),
        Cage(capacity=1, distance=70),
        Cage(capacity=8, distance=80),
        Cage(capacity=9, distance=90),
    ]

    total_discomfort, assignment, sorted_tigers, sorted_cages = min_discomfort(tigers, cages)

    print("Minimum total discomfort:", total_discomfort)
    print("\nAssignments (Tiger → Cage):")
    for i, cage_idx in enumerate(assignment):
        tiger = sorted_tigers[i]
        cage = sorted_cages[cage_idx]
        print(f"Tiger {i} (age={tiger.age}, size={tiger.size}) → "
              f"Cage {cage_idx} (capacity={cage.capacity}, distance={cage.distance})")


[[0, inf, inf, inf, inf, inf, inf, inf, inf, inf], [inf, 6, 4, 8, 5, 3, 7, 9, inf, inf], [inf, inf, 6, 5, 4, 4, 3, 5, 3, inf], [inf, inf, inf, 10, 6, 4, 7, 8, 3, 3]]
Minimum total discomfort: 3

Assignments (Tiger → Cage):
Tiger 0 (age=10, size=10) → Cage 4 (capacity=7, distance=50)
Tiger 1 (age=8, size=3) → Cage 5 (capacity=3, distance=60)
Tiger 2 (age=6, size=6) → Cage 7 (capacity=8, distance=80)
